# Transaction outcomes: change a ticket and its history together

Save a copy in Drive. This notebook runs a real PostgreSQL server inside the Colab runtime. It needs no cloud account or database password. The temporary database disappears when the runtime expires; the notebook and saved outputs remain in your Drive copy.

Required hand-in: `week_05_transaction_outcomes.sql`. Submit the SQL file in Brightspace with the lab’s comments and observed results. This notebook is an optional workspace, not an additional submission.

**Match the example:** The Day 1 lecture's worked assignment uses Agent 202 and event 5998. The student transaction lab deliberately uses Agent 201 and event 5999. These are separate rehearsal examples; use the IDs printed in the block you are running. The Day 2 lock_lab row begins medium/open, whereas the full Metro Support ticket 1004 begins low/new.


## Run order and submission

Run numbered cells in order and stop at each student workspace. The cell title is the number used in the teaching guide. Collapsed setup still runs; expand it to inspect the implementation.

| Cell | What you do |
|---|---|
| 01 | Install and locate PostgreSQL (body collapsed) |
| 02 | Create one practice database (body collapsed) |
| 03 | Define the result display helper (body collapsed) |
| 04 | Load the complete fixture once (body collapsed) |
| 05 | Check counts and ticket 1004 |
| 06 | Create the isolated transaction_lab copy |
| 07 | Rehearse both changes and roll back |
| 08 | Verify both changes were discarded |
| 09 | Commit the approved assignment |
| 10 | Observe the duplicate-key rejection |
| 11 | Verify the earlier commit survived the failed pair |
| 12 | Test an outdated assignment request |
| 13 | Optional cleanup of this practice database (body collapsed) |

Required hand-in: `week_05_transaction_outcomes.sql`. Keep your SQL and explanations in that file; no separate notebook submission.

## Complete data reference

Metro Support is a fictional help desk. All people and records are synthetic.
The complete lab fixture contains **8 users, 12 tickets, and 21 events**. A user
may request many tickets; a ticket may have one assignee or no assignee; a ticket
may have many history events. Requester, assignee, and actor refer to different
roles and all point to `users.user_id`.

These are the fixture's observed values, not all database-enforced rules. The
baseline SQL declares keys, required fields, unique email, and the date-order
check. It does **not** yet declare allowed-value CHECK constraints for status,
priority, category, or role. The integrity lab adds some of those rules.

Every row has the declared columns. SQL NULL is a value marker for missing or
inapplicable information, not an absent column, zero, or the text 'NULL'. Empty
CSV fields in this fixture load as SQL NULL. The SQL uses explicit integer IDs;
the identity experiment later creates a separate generated `note_id`.

## users

One row represents one person.

| Field | Type and rule | Meaning |
|---|---|---|
| `user_id` | integer; primary key | Unique person identifier; supplied explicitly in this fixture |
| `display_name` | text; required | Synthetic display label; it is not a stable join key |
| `email` | text; required and unique | Synthetic contact address ending in example.test |
| `role` | text; required | Observed: resident, agent, supervisor, analyst |
| `neighborhood` | text; required | Observed: Harbor, Northside, Central |
| `created_at` | timestamp with time zone; required | When the user record was created |

One complete example record (JSON notation for readability; SQL uses typed columns):

```json
{
  "user_id": 101,
  "display_name": "Requester 101",
  "email": "requester101@example.test",
  "role": "resident",
  "neighborhood": "Harbor",
  "created_at": "2026-01-08T14:20:00Z"
}
```

## tickets

One row represents one request.

| Field | Type and rule | Meaning |
|---|---|---|
| `ticket_id` | integer; primary key | One stable identifier for one request |
| `requester_id` | integer; required foreign key | Person who reported the request; references users.user_id |
| `assignee_id` | integer; nullable foreign key | Staff member assigned to the request; NULL means unassigned |
| `category` | text; required | Observed: parks, sanitation, streetlight, transportation, water |
| `priority` | text; required | Observed: low, medium, high, urgent |
| `status` | text; required | Observed: new, open, in_progress, resolved, closed |
| `subject` | text; required | Short description of the request |
| `opened_at` | timestamp with time zone; required | When the request was recorded |
| `closed_at` | timestamp with time zone; nullable | Closure time if recorded; NULL is not a date |

One complete example record (JSON notation for readability; SQL uses typed columns):

```json
{
  "ticket_id": 1001,
  "requester_id": 101,
  "assignee_id": 201,
  "category": "streetlight",
  "priority": "high",
  "status": "open",
  "subject": "Streetlight dark near bus stop",
  "opened_at": "2026-02-01T23:10:00Z",
  "closed_at": null
}
```

## ticket_events

One row represents one event on one request.

| Field | Type and rule | Meaning |
|---|---|---|
| `event_id` | integer; primary key | One stable identifier for one history event |
| `ticket_id` | integer; required foreign key | Request this event describes; references tickets.ticket_id |
| `actor_id` | integer; required foreign key | Person who acted; references users.user_id |
| `event_type` | text; required | Observed: created, assigned, status_changed |
| `old_status` | text; nullable | Prior status when relevant; absent for a creation event |
| `new_status` | text; nullable | Recorded resulting status when relevant |
| `note` | text; nullable | Explanation attached to the event |
| `event_at` | timestamp with time zone; required | When the event was recorded |

One complete example record (JSON notation for readability; SQL uses typed columns):

```json
{
  "event_id": 5001,
  "ticket_id": 1001,
  "actor_id": 101,
  "event_type": "created",
  "old_status": null,
  "new_status": "open",
  "note": "Reported through mobile form",
  "event_at": "2026-02-01T23:10:00Z"
}
```

## Names that change between exercises

- Week 1 uses a four-ticket extract and a two-row `agents` table. Its `agent_id`
  corresponds to the later `users.user_id`; its `name` corresponds to `display_name`.
- The first Week 2 examples use a smaller **4 / 6 / 9** fixture in `main`, with
  fewer columns. The separate **8 / 12 / 21** lab fixture uses `metro_support`.
  They are different instances; do not transfer requester IDs or counts between them.
- Week 5's `lock_lab.tickets` intentionally has only `ticket_id`, `priority`,
  and `status`. Its complete starting row is **1004 / medium / open**. The main
  Metro Support row 1004 is **low / new**. Always name the schema when explaining it.
- In the full fixture, active means `new`, `open`, or `in_progress`. Week 1's
  smaller question defines active as only `new` or `open`.

The original CSV files retain every record. The notebook also provides a named
fixture-loading cell and a visible count check before the lab queries.


## Setup once

Run the next four collapsed cells in order. They install PostgreSQL, create one uniquely named practice database, define a result display helper, and load the complete fixture. The visible count check after setup should show 8 users, 12 tickets, and 21 events. Do not rerun setup to solve a query error; your existing work belongs to this practice database.

In [ ]:
#@title Cell 01 · Install and locate PostgreSQL
import getpass
import os
from pathlib import Path
import subprocess
from uuid import uuid4

# Local Jupyter only: uncomment and supply your actual socket directory.
# os.environ["PGHOST"] = "/path/to/your/postgresql/socket"
# os.environ["PGPORT"] = "5432"

if "PRACTICE_DB" in globals():
    raise RuntimeError("Keep this database for both days, or run cleanup first.")
if os.environ.get("PGHOSTADDR") or os.environ.get("PGSERVICE"):
    raise RuntimeError("Unset PGHOSTADDR and PGSERVICE; use an explicit local socket.")

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "postgresql", "postgresql-client"], check=True)
    subprocess.run(["service", "postgresql", "start"], check=True)
    COLAB_SERVICE_STARTED = True

PGHOST = "/var/run/postgresql" if IN_COLAB else os.environ.get("PGHOST", "")
if not Path(PGHOST).is_absolute() or "," in PGHOST or not Path(PGHOST).is_dir():
    raise RuntimeError("Set PGHOST to one existing local Unix socket directory.")
PGPORT = "5432" if IN_COLAB else os.environ.get("PGPORT", "5432")
PGUSER = "postgres" if IN_COLAB else os.environ.get("PGUSER", getpass.getuser())
PG_PREFIX = ["sudo", "-u", "postgres"] if IN_COLAB else []
PG_ARGS = ["--host", PGHOST, "--port", PGPORT, "--username", PGUSER, "--no-password"]
subprocess.run(PG_PREFIX + ["pg_isready"] + PG_ARGS[:-1] + ["--dbname", "postgres"], check=True)
print("Local socket:", PGHOST, "Port:", PGPORT)

In [ ]:
#@title Cell 02 · Create one practice database
# template0 supplies an empty database, not a copy of your course work.
if "PRACTICE_DB" in globals():
    raise RuntimeError("Database already created. Do not reset it between days.")
run_id = uuid4().hex
new_database = "cst4714_transactions_" + run_id
subprocess.run(
    PG_PREFIX + ["createdb"] + PG_ARGS
    + ["--maintenance-db=postgres", "--template=template0", new_database],
    check=True,
)
PRACTICE_DB = new_database  # Remember only the database this run created.
print("This notebook owns:", PRACTICE_DB)

In [ ]:
#@title Cell 03 · Define the result display helper
def run_sql(sql_text):
    result = subprocess.run(
        PG_PREFIX + ["psql"] + PG_ARGS
        + ["-X", "--dbname", PRACTICE_DB, "--set=ON_ERROR_STOP=on",
           "--set=VERBOSITY=verbose", "--pset=null=NULL", "--pset=pager=off"],
        input=sql_text, text=True, capture_output=True,
    )
    print(result.stdout)
    if result.returncode:
        raise RuntimeError(result.stderr.strip())

In [ ]:
#@title Cell 04 · Load the complete fixture once
if globals().get("FIXTURE_LOADED", False):
    raise RuntimeError("Fixture already loaded. Keep your work; do not reset between days.")
assert PRACTICE_DB == "cst4714_transactions_" + run_id
setup_sql = """-- Metro Support PostgreSQL setup
-- Run only in a course or personal practice database. This resets the
-- metro_support schema so the dataset is reproducible.

DROP SCHEMA IF EXISTS metro_support CASCADE;
CREATE SCHEMA metro_support;
SET search_path TO metro_support, public;

CREATE TABLE users (
    user_id integer PRIMARY KEY,
    display_name text NOT NULL,
    email text NOT NULL UNIQUE,
    role text NOT NULL,
    neighborhood text NOT NULL,
    created_at timestamptz NOT NULL
);

CREATE TABLE tickets (
    ticket_id integer PRIMARY KEY,
    requester_id integer NOT NULL REFERENCES users(user_id),
    assignee_id integer REFERENCES users(user_id),
    category text NOT NULL,
    priority text NOT NULL,
    status text NOT NULL,
    subject text NOT NULL,
    opened_at timestamptz NOT NULL,
    closed_at timestamptz,
    CHECK (closed_at IS NULL OR closed_at >= opened_at)
);

CREATE TABLE ticket_events (
    event_id integer PRIMARY KEY,
    ticket_id integer NOT NULL REFERENCES tickets(ticket_id),
    actor_id integer NOT NULL REFERENCES users(user_id),
    event_type text NOT NULL,
    old_status text,
    new_status text,
    note text,
    event_at timestamptz NOT NULL
);

INSERT INTO users
    (user_id, display_name, email, role, neighborhood, created_at)
VALUES
    (101, 'Requester 101', 'requester101@example.test', 'resident', 'Harbor', '2026-01-08T14:20:00Z'),
    (102, 'Requester 102', 'requester102@example.test', 'resident', 'Northside', '2026-01-10T09:15:00Z'),
    (103, 'Requester 103', 'requester103@example.test', 'resident', 'Central', '2026-01-12T18:05:00Z'),
    (104, 'Requester 104', 'requester104@example.test', 'resident', 'Harbor', '2026-01-18T11:40:00Z'),
    (201, 'Agent 201', 'agent201@example.test', 'agent', 'Central', '2025-11-03T13:00:00Z'),
    (202, 'Agent 202', 'agent202@example.test', 'agent', 'Northside', '2025-11-05T13:00:00Z'),
    (203, 'Agent 203', 'agent203@example.test', 'supervisor', 'Central', '2025-09-14T13:00:00Z'),
    (204, 'Sam Okafor', 'sam.okafor@example.test', 'analyst', 'Harbor', '2025-12-01T13:00:00Z');

INSERT INTO tickets
    (ticket_id, requester_id, assignee_id, category, priority, status, subject, opened_at, closed_at)
VALUES
    (1001, 101, 201, 'streetlight', 'high', 'open', 'Streetlight dark near bus stop', '2026-02-01T23:10:00Z', NULL),
    (1002, 102, 202, 'sanitation', 'medium', 'in_progress', 'Missed recycling pickup', '2026-02-02T15:45:00Z', NULL),
    (1003, 103, 201, 'water', 'urgent', 'resolved', 'Low water pressure', '2026-02-03T12:05:00Z', '2026-02-03T19:40:00Z'),
    (1004, 104, NULL, 'parks', 'low', 'new', 'Broken bench slat', '2026-02-04T17:20:00Z', NULL),
    (1005, 101, 202, 'sanitation', 'high', 'resolved', 'Overflowing corner bin', '2026-02-05T14:00:00Z', '2026-02-05T20:15:00Z'),
    (1006, 102, 201, 'streetlight', 'medium', 'in_progress', 'Flickering lamp outside library', '2026-02-06T01:30:00Z', NULL),
    (1007, 103, 202, 'parks', 'medium', 'open', 'Playground gate will not latch', '2026-02-07T16:10:00Z', NULL),
    (1008, 104, 201, 'water', 'high', 'resolved', 'Hydrant leaking slowly', '2026-02-08T10:25:00Z', '2026-02-09T09:05:00Z'),
    (1009, 101, NULL, 'transportation', 'medium', 'new', 'Bus shelter panel cracked', '2026-02-09T22:15:00Z', NULL),
    (1010, 102, 202, 'sanitation', 'low', 'closed', 'Replacement bin request', '2026-02-10T13:50:00Z', '2026-02-12T16:30:00Z'),
    (1011, 103, 201, 'transportation', 'high', 'open', 'Crosswalk signal delayed', '2026-02-11T08:35:00Z', NULL),
    (1012, 104, 202, 'streetlight', 'low', 'resolved', 'Lamp stays on during daytime', '2026-02-12T14:45:00Z', '2026-02-14T18:10:00Z');

INSERT INTO ticket_events
    (event_id, ticket_id, actor_id, event_type, old_status, new_status, note, event_at)
VALUES
    (5001, 1001, 101, 'created', NULL, 'open', 'Reported through mobile form', '2026-02-01T23:10:00Z'),
    (5002, 1001, 201, 'assigned', 'open', 'open', 'Electrical crew notified', '2026-02-02T14:05:00Z'),
    (5003, 1002, 102, 'created', NULL, 'open', 'Pickup was scheduled for Monday', '2026-02-02T15:45:00Z'),
    (5004, 1002, 202, 'status_changed', 'open', 'in_progress', 'Route supervisor checking vehicle log', '2026-02-03T13:30:00Z'),
    (5005, 1003, 103, 'created', NULL, 'open', 'Pressure lower on two floors', '2026-02-03T12:05:00Z'),
    (5006, 1003, 201, 'status_changed', 'open', 'in_progress', 'Crew dispatched', '2026-02-03T14:25:00Z'),
    (5007, 1003, 201, 'status_changed', 'in_progress', 'resolved', 'Valve adjustment restored pressure', '2026-02-03T19:40:00Z'),
    (5008, 1004, 104, 'created', NULL, 'new', 'Photo attached in original report', '2026-02-04T17:20:00Z'),
    (5009, 1005, 101, 'created', NULL, 'open', 'Bin blocks part of sidewalk', '2026-02-05T14:00:00Z'),
    (5010, 1005, 202, 'status_changed', 'open', 'resolved', 'Extra collection completed', '2026-02-05T20:15:00Z'),
    (5011, 1006, 102, 'created', NULL, 'open', 'Flicker repeats every few seconds', '2026-02-06T01:30:00Z'),
    (5012, 1006, 201, 'status_changed', 'open', 'in_progress', 'Ballast inspection scheduled', '2026-02-06T15:10:00Z'),
    (5013, 1007, 103, 'created', NULL, 'open', 'Gate opens toward play area', '2026-02-07T16:10:00Z'),
    (5014, 1008, 104, 'created', NULL, 'open', 'Small stream along curb', '2026-02-08T10:25:00Z'),
    (5015, 1008, 201, 'status_changed', 'open', 'resolved', 'Gasket replaced and area checked', '2026-02-09T09:05:00Z'),
    (5016, 1009, 101, 'created', NULL, 'new', 'No sharp edge visible', '2026-02-09T22:15:00Z'),
    (5017, 1010, 102, 'created', NULL, 'open', 'Current bin lid is missing', '2026-02-10T13:50:00Z'),
    (5018, 1010, 202, 'status_changed', 'open', 'closed', 'Replacement delivered', '2026-02-12T16:30:00Z'),
    (5019, 1011, 103, 'created', NULL, 'open', 'Wait exceeds one full light cycle', '2026-02-11T08:35:00Z'),
    (5020, 1012, 104, 'created', NULL, 'open', 'Possible photocell issue', '2026-02-12T14:45:00Z'),
    (5021, 1012, 202, 'status_changed', 'open', 'resolved', 'Photocell cleaned and tested', '2026-02-14T18:10:00Z');

-- Verification: these counts should be 8, 12, and 21.
SELECT 'users' AS table_name, count(*) AS row_count FROM users
UNION ALL
SELECT 'tickets', count(*) FROM tickets
UNION ALL
SELECT 'ticket_events', count(*) FROM ticket_events;
"""
run_sql(setup_sql)
FIXTURE_LOADED = True

## Check the data before changing it

The schema `metro_support` holds the source fixture. Read the row and field reference above before using a name. The first query below checks table sizes; the next shows the complete ticket used in the transaction and integrity examples.

In [ ]:
#@title Cell 05 · Check counts and ticket 1004
run_sql("""
SELECT 'users' AS table_name, count(*) FROM metro_support.users
UNION ALL SELECT 'tickets', count(*) FROM metro_support.tickets
UNION ALL SELECT 'events', count(*) FROM metro_support.ticket_events;
SELECT * FROM metro_support.tickets WHERE ticket_id = 1004;
""")

## 1. Make an Isolated Practice Copy

The collapsed cells above created your private runtime database and loaded Metro Support. This next cell resets only the disposable `transaction_lab` practice copy.



In [ ]:
#@title Cell 06 · Create the isolated transaction_lab copy
run_sql("""
DROP SCHEMA IF EXISTS transaction_lab CASCADE;
CREATE SCHEMA transaction_lab;
CREATE TABLE transaction_lab.tickets AS
SELECT * FROM metro_support.tickets;
CREATE TABLE transaction_lab.ticket_events AS
SELECT * FROM metro_support.ticket_events;
ALTER TABLE transaction_lab.tickets ADD PRIMARY KEY (ticket_id);
ALTER TABLE transaction_lab.ticket_events ADD PRIMARY KEY (event_id);

SELECT ticket_id, assignee_id, status
FROM transaction_lab.tickets WHERE ticket_id = 1004;
""")



Expect ticket 1004 to be unassigned and `new`. CREATE TABLE AS copies query
results, not all source constraints; the two primary keys above are explicit.
These disposable tables are separate from the source data.

## 2. Rehearse One Assignment

The proposed assignee is agent **201**. Here is the complete state change:



In [ ]:
#@title Cell 07 · Rehearse both changes and roll back
run_sql("""
BEGIN;
UPDATE transaction_lab.tickets
SET assignee_id = 201, status = 'in_progress'
WHERE ticket_id = 1004 AND status = 'new'
RETURNING ticket_id, assignee_id, status;

INSERT INTO transaction_lab.ticket_events
    (event_id, ticket_id, actor_id, event_type, old_status, new_status, note, event_at)
VALUES
    (5999, 1004, 201, 'status_changed', 'new', 'in_progress',
     'Assigned to Agent 201 for follow-up', now());

SELECT ticket_id, assignee_id, status
FROM transaction_lab.tickets WHERE ticket_id = 1004;
SELECT event_id, ticket_id, new_status
FROM transaction_lab.ticket_events WHERE event_id = 5999;
ROLLBACK;
""")



Before running the whole block together, predict what the two SELECTs inside it
will show: ticket 1004 assigned to 201 and event 5999 present. Those are observations
inside the transaction. After it finishes, run these queries separately:



In [ ]:
#@title Cell 08 · Verify both changes were discarded
run_sql("""
SELECT ticket_id, assignee_id, status
FROM transaction_lab.tickets WHERE ticket_id = 1004;
SELECT event_id, ticket_id, new_status
FROM transaction_lab.ticket_events WHERE event_id = 5999;
""")



The ticket should be back to its starting state and event 5999 should be absent.

## 3. Keep the Approved Assignment

Run this complete transaction to keep the assignment:



In [ ]:
#@title Cell 09 · Commit the approved assignment
run_sql("""
BEGIN;
UPDATE transaction_lab.tickets
SET assignee_id = 201, status = 'in_progress'
WHERE ticket_id = 1004 AND status = 'new'
RETURNING ticket_id, assignee_id, status;

INSERT INTO transaction_lab.ticket_events
    (event_id, ticket_id, actor_id, event_type, old_status, new_status, note, event_at)
VALUES
    (5999, 1004, 201, 'status_changed', 'new', 'in_progress',
     'Assigned to Agent 201 for follow-up', now());
COMMIT;
""")



Run the two verification queries from step 2 again. Expect ticket 1004 assigned
to **201**, status **in_progress**, and event **5999** present. A COMMIT issued
after a completed ROLLBACK cannot recover the discarded changes; the UPDATE and
INSERT must run again in the new transaction above.

## 4. Observe a Failed Pair

Ticket 1004's original priority is `low`. Event **5001** already exists in the
copied setup data. This test attempts to insert that entire existing row again,
so the duplicate-key error does not depend on whether event 5999 was committed.



In [ ]:
#@title Cell 10 · Observe the duplicate-key rejection
try:
    run_sql("""
    BEGIN;
    UPDATE transaction_lab.tickets
    SET priority = 'urgent' WHERE ticket_id = 1004;
    INSERT INTO transaction_lab.ticket_events
    SELECT * FROM transaction_lab.ticket_events WHERE event_id = 5001;
    """)
except RuntimeError as error:
    if "23505" not in str(error):
        raise
    print("Expected duplicate-key rejection (23505):", error)
    print("This psql connection closed; its failed transaction was rolled back.")




Expect a duplicate-primary-key error naming **5001** (SQLSTATE `23505`).
After the error, run this cleanup and verification block separately:



In [ ]:
#@title Cell 11 · Verify the earlier commit survived the failed pair
run_sql("""
ROLLBACK;
SELECT ticket_id, assignee_id, status, priority
FROM transaction_lab.tickets WHERE ticket_id = 1004;
SELECT event_id, ticket_id, new_status
FROM transaction_lab.ticket_events
WHERE event_id IN (5001, 5999)
ORDER BY event_id;
""")



After completing step 3, expect **201, in_progress, low** for ticket 1004 and
one row each for events **5001** and **5999**. The failed transaction keeps neither
the new priority nor a duplicate event. It does not undo the assignment or event
that already committed. If testing this block before step 3, the duplicate error
still occurs, but event 5999 remains absent and the ticket remains unassigned.

## 5. Compare an Outdated Assignment Request

Complete step 3 before this test and verify its stored results. Agent 201 already
has the ticket, but another request still assumes its status is `new`. Predict
the two results, then run this whole block:



In [ ]:
#@title Cell 12 · Test an outdated assignment request
run_sql("""
BEGIN;
UPDATE transaction_lab.tickets
SET assignee_id = 202, status = 'in_progress'
WHERE ticket_id = 1004 AND status = 'new'
RETURNING ticket_id, assignee_id, status;

SELECT ticket_id, assignee_id, status
FROM transaction_lab.tickets WHERE ticket_id = 1004;
ROLLBACK;
""")



The UPDATE should return **no rows**, without an SQL error. The SELECT should
still show **201, in_progress**. We deliberately do not insert an event in this
probe. A program that announced an assignment to Agent 202 merely because no exception
occurred would report a change that never happened.

In your SQL comments, explain why the failed pair kept neither new change and
what could go wrong if its UPDATE committed separately. Then write one sentence
to the application developer: which result must the assignment code check before
it records history or tells the caller that the assignment succeeded? A zero-row
result calls for examining the current record; by itself it cannot distinguish
an outdated status from a missing ticket.



## Your explanation

Write your answers here, using the outputs you saved:

1. Which two changes disappeared after the rehearsal, and which two remained after commit?
2. Why did the duplicate-key error leave priority at its earlier value?
3. What must the application check before it records an assignment event?

**Your draft response (copy the required explanation into SQL comments for your hand-in):**


## Save your work before cleanup

Required hand-in: `week_05_transaction_outcomes.sql`. Submit the SQL file in Brightspace with the lab’s comments and observed results. This notebook is an optional workspace, not an additional submission.

Do not run cleanup until your results and written explanations are saved. Each call to `run_sql` uses a new PostgreSQL connection. Keep a whole `BEGIN` through `COMMIT` or `ROLLBACK` in one call; an unfinished transaction is rolled back when that connection closes.

## Optional cleanup

This removes only the uniquely named database created by this notebook. It does not delete your saved notebook.

In [ ]:
#@title Cell 13 · Optional cleanup of this practice database
if "PRACTICE_DB" in globals():
    assert PRACTICE_DB == "cst4714_transactions_" + run_id
    subprocess.run(
        PG_PREFIX + ["dropdb"] + PG_ARGS
        + ["--maintenance-db=postgres", "--if-exists", PRACTICE_DB],
        check=True,
    )
    print("Removed:", PRACTICE_DB)
    del PRACTICE_DB
    globals().pop("FIXTURE_LOADED", None)
if globals().get("COLAB_SERVICE_STARTED", False):
    subprocess.run(["service", "postgresql", "stop"], check=True)
    COLAB_SERVICE_STARTED = False
print("Cleanup complete. A local computer's server is left running.")